# Load and Normalise and Select relevent data

## Load

In [1]:
import subprocess
import pandas as pd
from pathlib import Path
import ast # Abstract Syntax Tree for dealing with dicts saved as strings. 
from dotenv import load_dotenv, find_dotenv
import os
import psycopg2
from psycopg2 import sql
import psycopg2.extras as extras
import json

load_dotenv(find_dotenv())

True

In [2]:
ROOT = Path(subprocess.check_output(["git", "rev-parse", "--show-toplevel"], text=True).strip())
BRONZE_PATH = ROOT / "Data" / "Bronze" / "enriched_scedule.parquet"
br_enriched_scedule = pd.read_parquet(BRONZE_PATH)
display(br_enriched_scedule.head(2))

,id,type,names,pronouns,title,description,short_description,video_privacy,is_fave,official_content,...,age_range,drop_in,family_friendly,content_note,LOAD_DATETIME,similarity_score,match_method,transcript_title,transcript,url
0,384,performance,"Field-FX organisers, DJ Deathboy",NaN,Demoscene Livecode Jam,We are going to code exciting visual effects i...,Visual livecoding jam with a DJ,public,False,False,...,NaN,None,None,NaN,2026-11-20 14:50:53.305170,NaN,NaN,NaN,NaN,NaN
1,165,workshop,Zoë Hutber,she/her,Aardman Modelling Workshops (all ages),Make your own Aardman character to take home! ...,Build your own clay model of an Aardman charac...,none,False,True,...,6+,False,True,NaN,2026-11-20 14:50:53.305170,NaN,NaN,NaN,NaN,NaN


## Drop irrelevent columns

In [3]:
br_enriched_scedule = br_enriched_scedule.drop(columns=["similarity_score","match_method","transcript_title", "slug", "link"]) # note link here is to position on website, the info there is contained in other cols. 
display(br_enriched_scedule.head(2))
display(br_enriched_scedule.columns)
assert not br_enriched_scedule.apply(lambda col: col.map(repr)).duplicated().any(), "Duplicate rows found." # Confirm Medalian Silver requirement of no duplicates. 

,id,type,names,pronouns,title,description,short_description,video_privacy,is_fave,official_content,occurrences,cost,equipment,age_range,drop_in,family_friendly,content_note,LOAD_DATETIME,transcript,url
0,384,performance,"Field-FX organisers, DJ Deathboy",NaN,Demoscene Livecode Jam,We are going to code exciting visual effects i...,Visual livecoding jam with a DJ,public,False,False,"[{'occurrence_num': 1, 'start_date': '2026-07-...",NaN,NaN,NaN,None,None,NaN,2026-11-20 14:50:53.305170,NaN,NaN
1,165,workshop,Zoë Hutber,she/her,Aardman Modelling Workshops (all ages),Make your own Aardman character to take home! ...,Build your own clay model of an Aardman charac...,none,False,True,"[{'occurrence_num': 1, 'start_date': '2026-07-...",£15,NaN,6+,False,True,NaN,2026-11-20 14:50:53.305170,NaN,NaN


Index(['id', 'type', 'names', 'pronouns', 'title', 'description',
       'short_description', 'video_privacy', 'is_fave', 'official_content',
       'occurrences', 'cost', 'equipment', 'age_range', 'drop_in',
       'family_friendly', 'content_note', 'LOAD_DATETIME', 'transcript',
       'url'],
      dtype='str')

## Safely normalise values that represent no cost with no additional info. 

In [4]:
# Normalising Costs. 
display(br_enriched_scedule.value_counts(["cost"]))

cost                                                                                                                          
free                                                                                                                              8
Free                                                                                                                              7
£15                                                                                                                               4
£10                                                                                                                               4
£1 refundable deposit to place items in the swap                                                                                  4
£20                                                                                                                               3
£5                                                                               

In [5]:
normalised_cost = pd.DataFrame({
    "original_cost": br_enriched_scedule["cost"],
    "normalised_cost": (
        br_enriched_scedule["cost"]
        .astype("string")
        .str.strip()
        .str.lower()
    ),
})

Zero_Synonyms = {"free", "nill", "nil", "no cost", "-", "£0",""}
zero_cost_mask = normalised_cost["normalised_cost"].isin(Zero_Synonyms)
normalised_cost.loc[zero_cost_mask, "normalised_cost"] = "£0"

display(normalised_cost["normalised_cost"].value_counts(dropna=False))

normalised_cost
<NA>                                                                                                                              483
£0                                                                                                                                 20
£15                                                                                                                                 4
£10                                                                                                                                 4
£1 refundable deposit to place items in the swap                                                                                    4
£20                                                                                                                                 3
£5                                                                                                                                  3
£22                                           

## Return normalised values back to the frame. 

In [6]:
br_enriched_scedule["cost"] = normalised_cost["normalised_cost"]
display(br_enriched_scedule["cost"])

0                                   <NA>
1                                    £15
2                                   <NA>
3                                   <NA>
4                                   <NA>
                     ...                
545                                   £0
546    £20 / £15 / £5 (pay what you can)
547                                 £375
548                                 <NA>
549                                 <NA>
Name: cost, Length: 551, dtype: string

# Extract Start and finish times from occurances and seperate each onto its own row. 

In [7]:
# ---------------------------------------------------------
# 1. IMPORT, VALIDATE AND IDENTIFY MULTIPLE OCCURRENCES
# ---------------------------------------------------------
br_enriched_scedule["occurrences"] = br_enriched_scedule["occurrences"].apply(
    lambda x: ast.literal_eval(x) if isinstance(x, str) else x
)
assert br_enriched_scedule["occurrences"].apply(
    lambda x: isinstance(x, list) and all(isinstance(i, dict) for i in x)
).all(), "Invalid occurrences structure."

br_enriched_scedule["multiple_occurrences"] = br_enriched_scedule[
    "occurrences"   
].str.len().gt(1).map({True: "Y", False: "N"})


# ---------------------------------------------------------
# 2. EXPAND SCHEDULE INTO INDIVIDUAL OCCURRENCE ROWS
# ---------------------------------------------------------
expanded_rows = [ # This pattern is nested list comprehension, It is a dense way to get all of the itterable parts in one line. It is equivalent to a nested for loop but more compact.
    {
        **row.drop( ## Drop the columns that will be replaced by occurrence-specific data
            ["occurrences", "start_date", "end_date", "occurrence_data"],
            errors="ignore" # Catches case where these columns don't exist in the DataFrame yet
        ).to_dict(),

        "source_row_index": index,
        "occurrence_data": occurrence.copy() if occurrence else None,
        "occurrence_number": number if occurrence else pd.NA,
        "occurrence_count": len(occurrences),
        "start_date": occurrence.get("start_date") if occurrence else pd.NaT,
        "end_date": occurrence.get("end_date") if occurrence else pd.NaT
    }

    for index, row in br_enriched_scedule.iterrows()
    for occurrences in [row["occurrences"]]
    for number, occurrence in enumerate(occurrences or [None], start=1)
]

expanded_schedule = pd.DataFrame(expanded_rows)

expanded_schedule[["start_date", "end_date"]] = expanded_schedule[
    ["start_date", "end_date"]
].apply(pd.to_datetime, errors="raise")


# ---------------------------------------------------------
# 3. CREATE MULTIPLE OCCURRENCES TROUBLESHOOTING DATAFRAME
# ---------------------------------------------------------
multiple_occurrences_df = expanded_schedule.query(
    "multiple_occurrences == 'Y'"
).copy()

# ---------------------------------------------------------
# 4. TRANSFORMATION RESULTS AND VALIDATION
# ---------------------------------------------------------
print("\n--- T2 OCCURRENCE EXPANSION RESULTS ---")

print(f"Original rows: {len(br_enriched_scedule)}")
print(f"Expanded rows: {len(expanded_schedule)}")
print(f"Additional rows: {len(expanded_schedule) - len(br_enriched_scedule)}")
print(f"Size of multiple occurrences DataFrame: {len(multiple_occurrences_df)}")

print("\n--- DATE COMPLETENESS ---")
for column in ["start_date", "end_date"]:
    print(f"{column}: {expanded_schedule[column].notna().sum()} populated | {expanded_schedule[column].isna().sum()} missing")
assert len(expanded_schedule) == sum(
    max(1, len(x)) for x in br_enriched_scedule["occurrences"]
)

assert pd.api.types.is_datetime64_any_dtype(expanded_schedule["start_date"])
assert pd.api.types.is_datetime64_any_dtype(expanded_schedule["end_date"])

assert (multiple_occurrences_df["multiple_occurrences"] == "Y").all()

print("\nTRANSFORMATION COMPLETE.")


# ---------------------------------------------------------
# 5. DISPLAY RESULTS AND EXPORT DTYPES
# ---------------------------------------------------------
Abridges_columns = ["id", "type","names","title","short_description","video_privacy","transcript",
    "multiple_occurrences","start_date","end_date","occurrence_data","occurrence_number",
    "occurrence_count"]

print("\n Display head of new expanded schedule.")
display(expanded_schedule[Abridges_columns].head(5))

print("\n Display rows with multiple occurrences for troubleshooting.")
display(multiple_occurrences_df[Abridges_columns].head(5))

print("\n showing dtypes of expanded schedule DataFrame:")
dtype_report = "\n".join(
    f"{i}. {col}: {dtype}"
    for i, (col, dtype) in enumerate(expanded_schedule.dtypes.items(), 1)
)

print("\n Missing date Troublehsooting")
display(
    expanded_schedule[expanded_schedule["start_date"].isna() | expanded_schedule["end_date"].isna()][Abridges_columns]
)

print(dtype_report)


--- T2 OCCURRENCE EXPANSION RESULTS ---
Original rows: 551
Expanded rows: 587
Additional rows: 36
Size of multiple occurrences DataFrame: 53

--- DATE COMPLETENESS ---
start_date: 585 populated | 2 missing
end_date: 585 populated | 2 missing

TRANSFORMATION COMPLETE.

 Display head of new expanded schedule.


,id,type,names,title,short_description,video_privacy,transcript,multiple_occurrences,start_date,end_date,occurrence_data,occurrence_number,occurrence_count
0,384,performance,"Field-FX organisers, DJ Deathboy",Demoscene Livecode Jam,Visual livecoding jam with a DJ,public,NaN,N,2026-07-17 19:30:00,2026-07-17 20:30:00,"{'occurrence_num': 1, 'start_date': '2026-07-1...",1,1
1,165,workshop,Zoë Hutber,Aardman Modelling Workshops (all ages),Build your own clay model of an Aardman charac...,none,NaN,Y,2026-07-18 13:00:00,2026-07-18 14:00:00,"{'occurrence_num': 1, 'start_date': '2026-07-1...",1,2
2,165,workshop,Zoë Hutber,Aardman Modelling Workshops (all ages),Build your own clay model of an Aardman charac...,none,NaN,Y,2026-07-18 15:00:00,2026-07-18 16:00:00,"{'occurrence_num': 2, 'start_date': '2026-07-1...",2,2
3,392,meetup,Mike Whisky,Mesh Networking Networking,Mesh networking fans meetup,public,NaN,N,2026-07-18 08:30:00,2026-07-18 09:00:00,"{'occurrence_num': 1, 'start_date': '2026-07-1...",1,1
4,82,talk,James Singleton,💚 Our Family's Electrotech Journey 🌞🏡🔌⛽️🚘️,"How we went all electric, ditched fossil fuels...",public,"[00:00] Thank you very much, everyone. Yeah, s...",N,2026-07-18 12:40:00,2026-07-18 13:10:00,"{'occurrence_num': 1, 'start_date': '2026-07-1...",1,1



 Display rows with multiple occurrences for troubleshooting.


,id,type,names,title,short_description,video_privacy,transcript,multiple_occurrences,start_date,end_date,occurrence_data,occurrence_number,occurrence_count
1,165,workshop,Zoë Hutber,Aardman Modelling Workshops (all ages),Build your own clay model of an Aardman charac...,none,NaN,Y,2026-07-18 13:00:00,2026-07-18 14:00:00,"{'occurrence_num': 1, 'start_date': '2026-07-1...",1,2
2,165,workshop,Zoë Hutber,Aardman Modelling Workshops (all ages),Build your own clay model of an Aardman charac...,none,NaN,Y,2026-07-18 15:00:00,2026-07-18 16:00:00,"{'occurrence_num': 2, 'start_date': '2026-07-1...",2,2
57,190,workshop,"Julie, Jon and Emily Spriggs",Board Game Tent,"We bring board games, or you bring board games...",public,NaN,Y,2026-07-18 15:00:00,2026-07-18 17:00:00,"{'occurrence_num': 1, 'start_date': '2026-07-1...",1,3
58,190,workshop,"Julie, Jon and Emily Spriggs",Board Game Tent,"We bring board games, or you bring board games...",public,NaN,Y,2026-07-18 18:30:00,2026-07-18 22:30:00,"{'occurrence_num': 2, 'start_date': '2026-07-1...",2,3
59,190,workshop,"Julie, Jon and Emily Spriggs",Board Game Tent,"We bring board games, or you bring board games...",public,NaN,Y,2026-07-19 15:00:00,2026-07-19 17:00:00,"{'occurrence_num': 3, 'start_date': '2026-07-1...",3,3



 showing dtypes of expanded schedule DataFrame:

 Missing date Troublehsooting


,id,type,names,title,short_description,video_privacy,transcript,multiple_occurrences,start_date,end_date,occurrence_data,occurrence_number,occurrence_count
259,175,performance,Karina Townsend,"DIY, modular bagpipes with sound activated las...","Modular, pneumatic sounds with rubber glove ba...",public,NaN,N,NaT,NaT,None,<NA>,0
427,554,talk,Sharon Kinsella,Infrastructure and Light,A talk about the core ideas in my oil painting...,public,NaN,N,NaT,NaT,None,<NA>,0


1. id: int64
2. type: str
3. names: str
4. pronouns: str
5. title: str
6. description: str
7. short_description: str
8. video_privacy: str
9. is_fave: bool
10. official_content: bool
11. cost: str
12. equipment: str
13. age_range: str
14. drop_in: object
15. family_friendly: object
16. content_note: str
17. LOAD_DATETIME: str
18. transcript: str
19. url: str
20. multiple_occurrences: str
21. source_row_index: int64
22. occurrence_data: object
23. occurrence_number: object
24. occurrence_count: int64
25. start_date: datetime64[us]
26. end_date: datetime64[us]


## Save to parquet to checkpoint progress. 

In [8]:
expanded_schedule["duration_min"] = (
    expanded_schedule["end_date"].sub(expanded_schedule["start_date"]).dt.total_seconds().div(60)
)

# PostgreSQL folds unquoted identifiers to lowercase
column_order = [
    "id",
    "type",
    "title",
    "short_description",
    "description",
    "names",
    "pronouns",
    "url",
    "transcript",
    "start_date",
    "end_date",
    "duration_min",
    "drop_in",
    "occurrence_count",
    "occurrence_number",
    "content_note",
    "family_friendly",
    "age_range",
    "cost",
    "equipment",
    "official_content",
    "video_privacy",
    "is_fave",
    "occurrence_data",
    "LOAD_DATETIME",
]

expanded_schedule = expanded_schedule.reindex(columns=[column for column in column_order if column in expanded_schedule.columns])
expanded_schedule = expanded_schedule.rename(columns={"LOAD_DATETIME": "BRONZE_LOAD_DATETIME"})
if "LOAD_DATETIME" in expanded_schedule.columns:
    expanded_schedule = expanded_schedule.rename(
        columns={"LOAD_DATETIME": "BRONZE_LOAD_DATETIME"}
    )

expanded_schedule["SILVER_LOAD_DATETIME"] = pd.Timestamp.now()

column_order = [
    column for column in column_order
    if column not in {"LOAD_DATETIME", "BRONZE_LOAD_DATETIME", "SILVER_LOAD_DATETIME"}
]
column_order += ["BRONZE_LOAD_DATETIME", "SILVER_LOAD_DATETIME"]

expanded_schedule = expanded_schedule.reindex(
    columns=[column for column in column_order if column in expanded_schedule.columns]
)

expanded_schedule.to_parquet(ROOT / "Data" / "Silver" / "sl_enriched_scedule")
display(pd.read_parquet(ROOT / "Data" / "Silver" / "sl_enriched_scedule").head(2))
display(expanded_schedule.columns)

,id,type,title,short_description,description,names,pronouns,url,transcript,start_date,...,family_friendly,age_range,cost,equipment,official_content,video_privacy,is_fave,occurrence_data,BRONZE_LOAD_DATETIME,SILVER_LOAD_DATETIME
0,384,performance,Demoscene Livecode Jam,Visual livecoding jam with a DJ,We are going to code exciting visual effects i...,"Field-FX organisers, DJ Deathboy",NaN,NaN,NaN,2026-07-17 19:30:00,...,None,NaN,NaN,NaN,False,public,False,"{'occurrence_num': 1.0, 'start_date': '2026-07...",2026-11-20 14:50:53.305170,2026-11-20 17:58:57.041549
1,165,workshop,Aardman Modelling Workshops (all ages),Build your own clay model of an Aardman charac...,Make your own Aardman character to take home! ...,Zoë Hutber,she/her,NaN,NaN,2026-07-18 13:00:00,...,True,6+,£15,NaN,True,none,False,"{'occurrence_num': 1.0, 'start_date': '2026-07...",2026-11-20 14:50:53.305170,2026-11-20 17:58:57.041549


Index(['id', 'type', 'title', 'short_description', 'description', 'names',
       'pronouns', 'url', 'transcript', 'start_date', 'end_date',
       'duration_min', 'drop_in', 'occurrence_count', 'occurrence_number',
       'content_note', 'family_friendly', 'age_range', 'cost', 'equipment',
       'official_content', 'video_privacy', 'is_fave', 'occurrence_data',
       'BRONZE_LOAD_DATETIME', 'SILVER_LOAD_DATETIME'],
      dtype='str')

# Load table to Postgres 

## Get Env Variables. 

In [9]:
CSV_FILE = os.getenv('CSV_FILE')
DB_NAME = os.getenv('DB_NAME')
DB_USER = os.getenv('DB_USER')
DB_PASSWORD = os.getenv('DB_PASSWORD')
DB_HOST = os.getenv('DB_HOST')
DB_PORT = os.getenv('DB_PORT')
TARGET_TABLE = "sl_enriched_schedule"

## Connect to Postgres DB

In [10]:
#Establish Connection to PostgreSQL Database
try:
    conn = psycopg2.connect(
        dbname=DB_NAME,
        user=DB_USER,
        password=DB_PASSWORD,
        host=DB_HOST,
        port=DB_PORT
    )
    print(f"Successfully connected to PostgreSQL database: {DB_NAME}")

except psycopg2.Error as e:
    print(f"Database connection failed. Error: {e}")
    print("Please check your database service and credentials.")

Successfully connected to PostgreSQL database: EMF_Camp


##  CREATE TABLE BLUEPRINT

In [11]:
expanded_schedule.info()

<class 'pandas.DataFrame'>
RangeIndex: 587 entries, 0 to 586
Data columns (total 26 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   id                    587 non-null    int64         
 1   type                  587 non-null    str           
 2   title                 587 non-null    str           
 3   short_description     501 non-null    str           
 4   description           587 non-null    str           
 5   names                 559 non-null    str           
 6   pronouns              236 non-null    str           
 7   url                   132 non-null    str           
 8   transcript            132 non-null    str           
 9   start_date            585 non-null    datetime64[us]
 10  end_date              585 non-null    datetime64[us]
 11  duration_min          585 non-null    float64       
 12  drop_in               176 non-null    object        
 13  occurrence_count      587 non-n

In [12]:
df_load = expanded_schedule # Make the syntax breifer. 

# Create the target table structure in PostgreSQL if it doesn't exist
create_table_query = sql.SQL(f"""
        CREATE TABLE IF NOT EXISTS {TARGET_TABLE}(
            id INTEGER UNIQUE,
            type VARCHAR(255),
            title VARCHAR(255),
            short_description TEXT,
            description TEXT,
            names VARCHAR(255),
            pronouns VARCHAR(100),
            url TEXT,  -- Confirm if URL has a specific type in PostgreSQL, otherwise use VARCHAR
            transcript TEXT,
            start_date TIMESTAMP WITH TIME ZONE,
            end_date TIMESTAMP WITH TIME ZONE,
            Duration_min FLOAT,
            drop_in BOOLEAN,
            occurrence_count INTEGER,
            occurrence_number INTEGER,
            content_note TEXT,
            family_friendly BOOLEAN,
            age_range VARCHAR(100),
            cost VARCHAR(100),
            equipment TEXT,
            official_content BOOLEAN,
            video_privacy VARCHAR(100),
            is_fave BOOLEAN,
            occurrence_data JSONB, -- Should confirm if this is the correct type. 
            BRONZE_LOAD_DATETIME TIMESTAMP WITH TIME ZONE DEFAULT CURRENT_TIMESTAMP,
            SILVER_LOAD_DATETIME TIMESTAMP WITH TIME ZONE DEFAULT CURRENT_TIMESTAMP
        );
    """
)

try:
    with conn.cursor() as cur:
        cur.execute(create_table_query)
        conn.commit()
    print(f"Table '{TARGET_TABLE}' already exists or has been successully created!")
except psycopg2.Error as e:
    print(f" Error creating table: {e}")
    conn.rollback()

Table 'sl_enriched_schedule' already exists or has been successully created!


## Insert data into table

In [13]:
# Quick diagnostic
str_cols = df_load.select_dtypes(include=["object", "string"]).columns
for col in str_cols:
    max_len = df_load[col].dropna().astype(str).str.len().max()
    if pd.notna(max_len) and max_len > 255:
        print(f"{col}: max length = {max_len}")

short_description: max length = 586
description: max length = 2814
transcript: max length = 98447
content_note: max length = 344
equipment: max length = 256
occurrence_data: max length = 715


In [14]:
def adapt_cell(val): # Helper function to handle case where column contains dictonaries. 
    if isinstance(val, dict):
        return extras.Json(val, dumps=lambda d: json.dumps(d, default=str))
    # Convert pandas/NumPy NaN/NaT to None so SQL gets NULL
    if val is None or (isinstance(val, float) and pd.isna(val)) or val is pd.NA:
        return None
    return val

# Convert the DataFrame to a list of tuples for psycopg2's execute_batch
data_tuples = [
    tuple(adapt_cell(val) for val in row)
    for row in df_load.itertuples(index=False)
]
# Define the INSERT query structure
cols = ', '.join(df_load.columns)
placeholders = ', '.join(['%s'] * len(df_load.columns))
# Define columns to update (all except the unique key 'order_id')
update_cols = [col for col in df_load.columns if col not in ['id']]
set_clause = ', '.join([f"{col} = EXCLUDED.{col}" for col in update_cols])
insert_query = sql.SQL(
    f"""
    INSERT INTO {TARGET_TABLE} ({cols}) 
    VALUES ({placeholders}) 
    ON CONFLICT (id) 
    DO UPDATE SET {set_clause}
    """
)
try:
    with conn.cursor() as cur:
        # Use execute_batch for efficient bulk insertion
        extras.execute_batch(cur, insert_query, data_tuples)
        conn.commit()
        print(f"Successfully attempted to load {len(data_tuples)} records into '{TARGET_TABLE}'. Conflicting records were updated.")

except psycopg2.Error as e:
    print(f"Error during data loading: {e}")
    conn.rollback()

finally:
    if conn:
        conn.close()
        print(" Database connection closed.")

print("\n--- ETL Process Complete ---")

Error during data loading: value too long for type character varying(100)

 Database connection closed.

--- ETL Process Complete ---
